In [0]:
# import libraries

import boto3
import pandas as pd
import re

In [0]:
# connect to S3 and find assessment files

s3 = boto3.client("s3")

paginator = s3.get_paginator("list_objects_v2")

assessment_files = [
    obj["Key"]
    for page in paginator.paginate(Bucket=bucket_name)
    for obj in page.get("Contents", [])
    if obj["Key"].startswith("Talent/")
    and obj["Key"].endswith(".txt")
]

In [0]:
# clean and transform assessment data

rows = []

for file in assessment_files:
    obj = s3.get_object(Bucket=bucket_name, Key=file)
    body = obj["Body"].read().decode("utf-8")
    lines = body.splitlines()

    assessment_date = None
    academy_location = None

    for line in lines:
        line = line.strip()

        if line == "":
            continue

        if re.search(r"\w+ \d{1,2} \w+ \d{4}", line):
            assessment_date = line
            continue

        if "Academy" in line:
            academy_location = line
            continue

        if "Psychometrics:" in line and "Presentation:" in line:
            match = re.match(
                r"(.+?)\s+-\s+Psychometrics:\s+(\d+)/100,\s+Presentation:\s+(\d+)/32",
                line
            )

            if match:
                rows.append({
                    "source_file": file,
                    "assessment_date": assessment_date,
                    "academy_location": academy_location,
                    "candidate_name": match.group(1).title(),
                    "psychometrics_score": int(match.group(2)),
                    "presentation_score": int(match.group(3))
                })

In [0]:
# create silver assessments table

silver_assessments = pd.DataFrame(rows)

spark_df = spark.createDataFrame(silver_assessments)

spark_df.write.mode("overwrite").saveAsTable("silver_assessments")

display(spark.table("silver_assessments"))

source_file,assessment_date,academy_location,candidate_name,psychometrics_score,presentation_score
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Phyllys Baelde,69,24
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Cecile Lates,55,22
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Charlean Devons,52,14
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Avrit Gawith,59,23
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Doralin Purkess,49,24
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Nikolos Yashin,57,20
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Emmanuel Deniske,50,16
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Mella Aubin,51,22
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Cal Loache,53,20
Talent/Sparta Day 15 August 2019.txt,Thursday 15 August 2019,Birmingham Academy,Salaidh Loveday,48,16
